# 07_03A — Walk-forward Baseline Early-Stage — CORRIGIDO

## Correção crítica do target

Neste projeto:

```text
target_banco_ganhou = 0 → banco ganhou / êxito
target_banco_ganhou = 1 → banco perdeu / condenação
```

Portanto, apesar do nome legado da coluna, a classe positiva `1` representa **risco de perda do banco**.

Neste notebook:

```text
proba_perda = predict_proba(...)[1]
score_risco_perda = proba_perda
```

Este notebook corrige:

1. nomes de taxas (`taxa_perda`, `taxa_ganho`);
2. PR-AUC como PR-AUC de perda;
3. top-k por risco de perda usando `proba_perda`, não `1 - proba`;
4. ranking por risco de perda;
5. ranking por prioridade financeira: `proba_perda × valor_ajuizado`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

pd.set_option("display.max_columns", 400)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 260)

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports" / "modelagem_early_stage"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DEV_HIST_FILE = PROCESSED_DIR / "abt_early_stage_v1_dev_hist.parquet"
HOLDOUT_HIST_FILE = PROCESSED_DIR / "abt_early_stage_v1_holdout_hist.parquet"
FEATURE_LIST_HIST_FILE = PROCESSED_DIR / "feature_list_early_stage_v1_hist.txt"

TARGET_COL = "target_banco_ganhou"  # nome legado: 0=ganhou, 1=perdeu
DATE_COL = "Datainicial"
VALUE_COL = "fe_valor_ajuizado"
RANDOM_STATE = 42

print("Setup concluído.")
print("Target: 0=banco_ganhou | 1=banco_perdeu")

## 2. Carregar bases

In [ ]:
df_dev = pd.read_parquet(DEV_HIST_FILE)
df_holdout = pd.read_parquet(HOLDOUT_HIST_FILE)

with open(FEATURE_LIST_HIST_FILE, "r", encoding="utf-8") as f:
    feature_list = [line.strip() for line in f if line.strip()]

df_dev[DATE_COL] = pd.to_datetime(df_dev[DATE_COL], errors="coerce")
df_holdout[DATE_COL] = pd.to_datetime(df_holdout[DATE_COL], errors="coerce")

df_dev[TARGET_COL] = df_dev[TARGET_COL].astype(int)
df_holdout[TARGET_COL] = df_holdout[TARGET_COL].astype(int)

feature_list = [c for c in feature_list if c in df_dev.columns and c in df_holdout.columns]

print("Dev:", df_dev.shape)
print("Holdout:", df_holdout.shape)
print("Features:", len(feature_list))
df_dev.head()

## 3. Validar target

In [ ]:
def target_distribution(df, name):
    out = (
        df[TARGET_COL]
        .value_counts(dropna=False)
        .rename_axis("target")
        .reset_index(name="qtd")
    )
    out["classe"] = out["target"].map({0: "banco_ganhou", 1: "banco_perdeu"})
    out["perc"] = out["qtd"] / out["qtd"].sum()
    out["dataset"] = name
    return out

target_dist = pd.concat([
    target_distribution(df_dev, "dev"),
    target_distribution(df_holdout, "holdout"),
], ignore_index=True)

target_dist.to_csv(REPORTS_DIR / "18_target_distribution_corrigido.csv", index=False)
target_dist

## 4. Funções auxiliares

In [ ]:
def save_report(df_report, filename):
    path = REPORTS_DIR / filename
    df_report.to_csv(path, index=False)
    print(f"Relatório salvo em: {path}")


def infer_feature_types(df, features):
    numeric_features, categorical_features, text_features = [], [], []

    for col in features:
        if col == "fe_texto_inicial_curto":
            text_features.append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            numeric_features.append(col)
        else:
            categorical_features.append(col)

    return numeric_features, categorical_features, text_features


def make_temporal_folds(df, date_col=DATE_COL):
    folds = [(0.55, 0.70), (0.70, 0.85), (0.85, 1.00)]
    df_sorted = df.sort_values(date_col).copy()
    dates = df_sorted[date_col]
    split_rows = []

    for i, (train_end_q, val_end_q) in enumerate(folds, start=1):
        train_end_date = dates.quantile(train_end_q)
        val_end_date = dates.quantile(val_end_q)

        train_idx = df_sorted.index[df_sorted[date_col] <= train_end_date]
        val_idx = df_sorted.index[
            (df_sorted[date_col] > train_end_date) &
            (df_sorted[date_col] <= val_end_date)
        ]

        split_rows.append({
            "fold": i,
            "train_start_date": df_sorted.loc[train_idx, date_col].min(),
            "train_end_date": train_end_date,
            "val_start_date": df_sorted.loc[val_idx, date_col].min(),
            "val_end_date": val_end_date,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "qtd_train": len(train_idx),
            "qtd_val": len(val_idx),
        })

    return split_rows


def threshold_metrics_perda(y_true, proba_perda, threshold=0.5):
    pred = (proba_perda >= threshold).astype(int)
    return {
        "threshold": threshold,
        "precision_perda": precision_score(y_true, pred, zero_division=0),
        "recall_perda": recall_score(y_true, pred, zero_division=0),
        "f1_perda": f1_score(y_true, pred, zero_division=0),
        "pred_perda_rate": pred.mean(),
    }


def find_best_f1_threshold_perda(y_true, proba_perda):
    precision, recall, thresholds = precision_recall_curve(y_true, proba_perda)
    if len(thresholds) == 0:
        return 0.5, 0, 0, 0

    precision_t = precision[:-1]
    recall_t = recall[:-1]
    f1_t = 2 * precision_t * recall_t / np.maximum(precision_t + recall_t, 1e-12)
    best_idx = np.nanargmax(f1_t)
    return thresholds[best_idx], precision_t[best_idx], recall_t[best_idx], f1_t[best_idx]


def topk_risco_perda_metrics(y_true, proba_perda, ks=(0.01, 0.05, 0.10, 0.20)):
    df_tmp = pd.DataFrame({
        "y_true": y_true,
        "score_risco_perda": proba_perda
    }).sort_values("score_risco_perda", ascending=False).reset_index(drop=True)

    total_perdas = (df_tmp["y_true"] == 1).sum()
    taxa_perda_base = (df_tmp["y_true"] == 1).mean()
    out = []

    for k in ks:
        n = max(1, int(np.ceil(len(df_tmp) * k)))
        top = df_tmp.head(n)

        precision_perda_at_k = (top["y_true"] == 1).mean()
        recall_perda_at_k = (top["y_true"] == 1).sum() / total_perdas if total_perdas else np.nan
        lift_perda_at_k = precision_perda_at_k / taxa_perda_base if taxa_perda_base else np.nan

        out.append({
            "top_k": k,
            "n_top": n,
            "precision_perda_at_k": precision_perda_at_k,
            "recall_perda_at_k": recall_perda_at_k,
            "lift_perda_at_k": lift_perda_at_k,
            "taxa_perda_base": taxa_perda_base,
        })

    return pd.DataFrame(out)


def topk_prioridade_financeira_metrics(y_true, proba_perda, valor_ajuizado, ks=(0.01, 0.05, 0.10, 0.20)):
    df_tmp = pd.DataFrame({
        "y_true": y_true,
        "proba_perda": proba_perda,
        "valor_ajuizado": pd.to_numeric(valor_ajuizado, errors="coerce").fillna(0),
    })
    df_tmp["score_prioridade_financeira"] = df_tmp["proba_perda"] * df_tmp["valor_ajuizado"]
    df_tmp = df_tmp.sort_values("score_prioridade_financeira", ascending=False).reset_index(drop=True)

    total_perdas = (df_tmp["y_true"] == 1).sum()
    taxa_perda_base = (df_tmp["y_true"] == 1).mean()
    total_valor = df_tmp["valor_ajuizado"].sum()
    total_valor_perdas = df_tmp.loc[df_tmp["y_true"] == 1, "valor_ajuizado"].sum()
    out = []

    for k in ks:
        n = max(1, int(np.ceil(len(df_tmp) * k)))
        top = df_tmp.head(n)

        valor_top = top["valor_ajuizado"].sum()
        valor_perdas_top = top.loc[top["y_true"] == 1, "valor_ajuizado"].sum()

        precision_perda_at_k = (top["y_true"] == 1).mean()
        recall_perda_at_k = (top["y_true"] == 1).sum() / total_perdas if total_perdas else np.nan
        lift_perda_at_k = precision_perda_at_k / taxa_perda_base if taxa_perda_base else np.nan

        out.append({
            "top_k": k,
            "n_top": n,
            "precision_perda_at_k": precision_perda_at_k,
            "recall_perda_at_k": recall_perda_at_k,
            "lift_perda_at_k": lift_perda_at_k,
            "taxa_perda_base": taxa_perda_base,
            "valor_ajuizado_top": valor_top,
            "share_valor_ajuizado_total": valor_top / total_valor if total_valor else np.nan,
            "valor_ajuizado_perdas_top": valor_perdas_top,
            "share_valor_perdas_total": valor_perdas_top / total_valor_perdas if total_valor_perdas else np.nan,
        })

    return pd.DataFrame(out)

## 5. Auditoria de features

In [ ]:
numeric_features, categorical_features, text_features = infer_feature_types(df_dev, feature_list)

feature_type_summary = pd.DataFrame([
    {"tipo": "numeric", "qtd": len(numeric_features)},
    {"tipo": "categorical", "qtd": len(categorical_features)},
    {"tipo": "text", "qtd": len(text_features)},
])

feature_audit = pd.DataFrame({
    "feature": feature_list,
    "tipo": [
        "numeric" if f in numeric_features else
        "categorical" if f in categorical_features else
        "text"
        for f in feature_list
    ],
    "dtype_dev": [str(df_dev[f].dtype) for f in feature_list],
    "missing_rate_dev": [df_dev[f].isna().mean() for f in feature_list],
    "missing_rate_holdout": [df_holdout[f].isna().mean() for f in feature_list],
    "nunique_dev": [df_dev[f].nunique(dropna=True) for f in feature_list],
    "nunique_holdout": [df_holdout[f].nunique(dropna=True) for f in feature_list],
})

save_report(feature_audit, "19_walk_forward_feature_audit_corrigido.csv")
display(feature_type_summary)
display(feature_audit.head(50))

## 6. Folds temporais

In [ ]:
folds = make_temporal_folds(df_dev, date_col=DATE_COL)

fold_summary_rows = []
for fold in folds:
    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]

    taxa_perda_train = df_dev.loc[train_idx, TARGET_COL].mean()
    taxa_perda_val = df_dev.loc[val_idx, TARGET_COL].mean()

    fold_summary_rows.append({
        "fold": fold["fold"],
        "train_start_date": fold["train_start_date"],
        "train_end_date": fold["train_end_date"],
        "val_start_date": fold["val_start_date"],
        "val_end_date": fold["val_end_date"],
        "qtd_train": fold["qtd_train"],
        "qtd_val": fold["qtd_val"],
        "taxa_perda_train": taxa_perda_train,
        "taxa_perda_val": taxa_perda_val,
        "taxa_ganho_train": 1 - taxa_perda_train,
        "taxa_ganho_val": 1 - taxa_perda_val,
    })

fold_summary = pd.DataFrame(fold_summary_rows)
save_report(fold_summary, "20_walk_forward_folds_summary_corrigido.csv")
fold_summary

## 7. Preprocessador e modelos

In [ ]:
def make_preprocessor(
    numeric_features,
    categorical_features,
    text_features,
    tfidf_max_features=1500,
    tfidf_min_df=20,
    onehot_min_frequency=20,
    scale_numeric=True,
):
    transformers = []

    if scale_numeric:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
    else:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="nao_informado")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=onehot_min_frequency)),
    ])

    if numeric_features:
        transformers.append(("num", numeric_pipeline, numeric_features))

    if categorical_features:
        transformers.append(("cat", categorical_pipeline, categorical_features))

    if text_features:
        text_col = text_features[0]
        text_pipeline = Pipeline([
            ("selector", FunctionTransformer(lambda x: x[text_col].fillna("").astype(str), validate=False)),
            ("tfidf", TfidfVectorizer(max_features=tfidf_max_features, ngram_range=(1, 2), min_df=tfidf_min_df)),
        ])
        transformers.append(("txt", text_pipeline, [text_col]))

    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3)


def make_logistic_pipeline():
    return Pipeline([
        ("preprocessor", make_preprocessor(numeric_features, categorical_features, text_features, 1500, 20, 20, True)),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", solver="saga", n_jobs=-1, random_state=RANDOM_STATE)),
    ])


def make_random_forest_pipeline():
    return Pipeline([
        ("preprocessor", make_preprocessor(numeric_features, categorical_features, text_features, 1000, 30, 30, False)),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=18,
            min_samples_split=30,
            min_samples_leaf=15,
            max_features="sqrt",
            max_samples=0.8,
            class_weight="balanced",
            criterion="entropy",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])


def make_hist_gradient_pipeline():
    hist_numeric_features = [c for c in numeric_features if c in df_dev.columns]
    preprocessor = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), hist_numeric_features)
    ], remainder="drop")

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=RANDOM_STATE,
        )),
    ])


model_factories = {
    "logistic_onehot_tfidf": make_logistic_pipeline,
    "random_forest_onehot_tfidf": make_random_forest_pipeline,
    "hist_gradient_numeric_only": make_hist_gradient_pipeline,
}

model_factories.keys()

## 8. Rodar walk-forward corrigido

In [ ]:
def evaluate_model_on_fold(model_name, pipeline, df, train_idx, val_idx, features):
    X_train = df.loc[train_idx, features].copy()
    y_train = df.loc[train_idx, TARGET_COL].copy()

    X_val = df.loc[val_idx, features].copy()
    y_val = df.loc[val_idx, TARGET_COL].copy()

    pipeline.fit(X_train, y_train)

    # Classe positiva 1 = banco perdeu.
    proba_perda_val = pipeline.predict_proba(X_val)[:, 1]

    roc_auc_perda = roc_auc_score(y_val, proba_perda_val)
    pr_auc_perda = average_precision_score(y_val, proba_perda_val)

    best_thr, best_p, best_r, best_f1 = find_best_f1_threshold_perda(y_val, proba_perda_val)
    thr_05 = threshold_metrics_perda(y_val, proba_perda_val, threshold=0.5)

    result = {
        "model": model_name,
        "roc_auc_perda": roc_auc_perda,
        "pr_auc_perda": pr_auc_perda,
        "taxa_perda_val": y_val.mean(),
        "taxa_ganho_val": 1 - y_val.mean(),
        "best_f1_threshold_perda": best_thr,
        "best_f1_precision_perda": best_p,
        "best_f1_recall_perda": best_r,
        "best_f1_perda": best_f1,
        "precision_perda_t05": thr_05["precision_perda"],
        "recall_perda_t05": thr_05["recall_perda"],
        "f1_perda_t05": thr_05["f1_perda"],
        "pred_perda_rate_t05": thr_05["pred_perda_rate"],
    }

    topk_perda = topk_risco_perda_metrics(y_val.values, proba_perda_val)

    if VALUE_COL in df.columns:
        topk_financeiro = topk_prioridade_financeira_metrics(
            y_true=y_val.values,
            proba_perda=proba_perda_val,
            valor_ajuizado=df.loc[val_idx, VALUE_COL].values
        )
    else:
        topk_financeiro = pd.DataFrame()

    return result, topk_perda, topk_financeiro


walk_forward_results = []
topk_perda_results = []
topk_financeiro_results = []

for model_name, factory in model_factories.items():
    print("=" * 100)
    print(f"Modelo: {model_name}")

    for fold in folds:
        print(f"  Fold {fold['fold']}")
        pipeline = factory()

        result, topk_perda, topk_financeiro = evaluate_model_on_fold(
            model_name=model_name,
            pipeline=pipeline,
            df=df_dev,
            train_idx=fold["train_idx"],
            val_idx=fold["val_idx"],
            features=feature_list,
        )

        result["fold"] = fold["fold"]
        result["qtd_train"] = fold["qtd_train"]
        result["qtd_val"] = fold["qtd_val"]
        result["train_end_date"] = fold["train_end_date"]
        result["val_start_date"] = fold["val_start_date"]
        result["val_end_date"] = fold["val_end_date"]

        walk_forward_results.append(result)

        topk_perda["model"] = model_name
        topk_perda["fold"] = fold["fold"]
        topk_perda_results.append(topk_perda)

        if not topk_financeiro.empty:
            topk_financeiro["model"] = model_name
            topk_financeiro["fold"] = fold["fold"]
            topk_financeiro_results.append(topk_financeiro)

walk_forward_results = pd.DataFrame(walk_forward_results)
topk_perda_results = pd.concat(topk_perda_results, ignore_index=True)
topk_financeiro_results = pd.concat(topk_financeiro_results, ignore_index=True) if topk_financeiro_results else pd.DataFrame()

save_report(walk_forward_results, "21_walk_forward_baseline_metrics_corrigido.csv")
save_report(topk_perda_results, "22_walk_forward_topk_risco_perda_metrics_corrigido.csv")

if not topk_financeiro_results.empty:
    save_report(topk_financeiro_results, "23_walk_forward_topk_prioridade_financeira_corrigido.csv")

walk_forward_results.sort_values(["pr_auc_perda", "roc_auc_perda"], ascending=False)

## 9. Resumo por modelo

In [ ]:
model_summary = (
    walk_forward_results
    .groupby("model")
    .agg(
        mean_roc_auc_perda=("roc_auc_perda", "mean"),
        std_roc_auc_perda=("roc_auc_perda", "std"),
        mean_pr_auc_perda=("pr_auc_perda", "mean"),
        std_pr_auc_perda=("pr_auc_perda", "std"),
        mean_taxa_perda_val=("taxa_perda_val", "mean"),
        mean_taxa_ganho_val=("taxa_ganho_val", "mean"),
        mean_best_f1_perda=("best_f1_perda", "mean"),
        mean_best_f1_threshold_perda=("best_f1_threshold_perda", "mean"),
        mean_precision_perda_t05=("precision_perda_t05", "mean"),
        mean_recall_perda_t05=("recall_perda_t05", "mean"),
        mean_f1_perda_t05=("f1_perda_t05", "mean"),
    )
    .reset_index()
    .sort_values("mean_pr_auc_perda", ascending=False)
)

save_report(model_summary, "24_walk_forward_model_summary_corrigido.csv")
model_summary

## 10. Top-k de risco de perda por modelo

In [ ]:
topk_perda_summary = (
    topk_perda_results
    .groupby(["model", "top_k"])
    .agg(
        mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
        std_precision_perda_at_k=("precision_perda_at_k", "std"),
        mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
        mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
        mean_taxa_perda_base=("taxa_perda_base", "mean"),
    )
    .reset_index()
    .sort_values(["top_k", "mean_precision_perda_at_k"], ascending=[True, False])
)

save_report(topk_perda_summary, "25_walk_forward_topk_risco_perda_summary_corrigido.csv")
topk_perda_summary

## 11. Top-k por prioridade financeira no walk-forward

In [ ]:
if not topk_financeiro_results.empty:
    topk_financeiro_summary = (
        topk_financeiro_results
        .groupby(["model", "top_k"])
        .agg(
            mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
            mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
            mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
            mean_share_valor_ajuizado_total=("share_valor_ajuizado_total", "mean"),
            mean_share_valor_perdas_total=("share_valor_perdas_total", "mean"),
        )
        .reset_index()
        .sort_values(["top_k", "mean_share_valor_perdas_total"], ascending=[True, False])
    )

    save_report(topk_financeiro_summary, "26_walk_forward_topk_prioridade_financeira_summary_corrigido.csv")
    display(topk_financeiro_summary)
else:
    print("Top-k financeiro não calculado.")

## 12. Treinar melhor modelo no Dev completo e avaliar Holdout

In [ ]:
best_model_name = model_summary.iloc[0]["model"]
print("Melhor modelo por PR-AUC de perda:", best_model_name)

best_pipeline = model_factories[best_model_name]()

X_dev = df_dev[feature_list].copy()
y_dev = df_dev[TARGET_COL].copy()

X_holdout = df_holdout[feature_list].copy()
y_holdout = df_holdout[TARGET_COL].copy()

best_pipeline.fit(X_dev, y_dev)

# Classe positiva 1 = banco perdeu.
proba_perda_holdout = best_pipeline.predict_proba(X_holdout)[:, 1]

holdout_roc_auc_perda = roc_auc_score(y_holdout, proba_perda_holdout)
holdout_pr_auc_perda = average_precision_score(y_holdout, proba_perda_holdout)

best_thr, best_p, best_r, best_f1 = find_best_f1_threshold_perda(y_holdout, proba_perda_holdout)
thr_05 = threshold_metrics_perda(y_holdout, proba_perda_holdout, threshold=0.5)

holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "roc_auc_perda": holdout_roc_auc_perda,
    "pr_auc_perda": holdout_pr_auc_perda,
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "best_f1_threshold_perda": best_thr,
    "best_f1_precision_perda": best_p,
    "best_f1_recall_perda": best_r,
    "best_f1_perda": best_f1,
    "precision_perda_t05": thr_05["precision_perda"],
    "recall_perda_t05": thr_05["recall_perda"],
    "f1_perda_t05": thr_05["f1_perda"],
    "pred_perda_rate_t05": thr_05["pred_perda_rate"],
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "qtd_features": len(feature_list),
}])

save_report(holdout_metrics, "27_holdout_best_baseline_metrics_corrigido.csv")
holdout_metrics

## 13. Top-k no Holdout por risco de perda

In [ ]:
holdout_topk_perda = topk_risco_perda_metrics(
    y_true=y_holdout.values,
    proba_perda=proba_perda_holdout,
    ks=(0.01, 0.05, 0.10, 0.20)
)

holdout_topk_perda["model"] = best_model_name

save_report(holdout_topk_perda, "28_holdout_topk_risco_perda_metrics_corrigido.csv")
holdout_topk_perda

## 14. Top-k no Holdout por prioridade financeira

In [ ]:
if VALUE_COL in df_holdout.columns:
    holdout_topk_financeiro = topk_prioridade_financeira_metrics(
        y_true=y_holdout.values,
        proba_perda=proba_perda_holdout,
        valor_ajuizado=df_holdout[VALUE_COL].values,
        ks=(0.01, 0.05, 0.10, 0.20)
    )

    holdout_topk_financeiro["model"] = best_model_name

    save_report(holdout_topk_financeiro, "29_holdout_topk_prioridade_financeira_metrics_corrigido.csv")
    display(holdout_topk_financeiro)
else:
    print(f"{VALUE_COL} não encontrado.")

## 15. Ranking corrigido do holdout

In [ ]:
ranking_cols = ["Numerodistribuicao", "Identificador", DATE_COL, TARGET_COL, VALUE_COL]
ranking_cols = [c for c in ranking_cols if c in df_holdout.columns]

ranking_holdout = df_holdout[ranking_cols].copy()
ranking_holdout["target_classe_real"] = ranking_holdout[TARGET_COL].map({
    0: "banco_ganhou",
    1: "banco_perdeu",
})

ranking_holdout["proba_banco_perdeu"] = proba_perda_holdout
ranking_holdout["score_risco_perda"] = proba_perda_holdout

if VALUE_COL in ranking_holdout.columns:
    ranking_holdout["score_prioridade_financeira"] = (
        ranking_holdout["score_risco_perda"] *
        pd.to_numeric(ranking_holdout[VALUE_COL], errors="coerce").fillna(0)
    )
else:
    ranking_holdout["score_prioridade_financeira"] = np.nan

ranking_holdout["rank_risco_perda"] = ranking_holdout["score_risco_perda"].rank(
    ascending=False,
    method="first"
).astype(int)

ranking_holdout["rank_prioridade_financeira"] = ranking_holdout["score_prioridade_financeira"].rank(
    ascending=False,
    method="first"
).astype(int)

ranking_risco = ranking_holdout.sort_values("score_risco_perda", ascending=False)
ranking_financeiro = ranking_holdout.sort_values("score_prioridade_financeira", ascending=False)

ranking_risco_path = PROCESSED_DIR / "ranking_holdout_risco_perda_baseline_early_stage_corrigido.parquet"
ranking_financeiro_path = PROCESSED_DIR / "ranking_holdout_prioridade_financeira_baseline_early_stage_corrigido.parquet"

ranking_risco.to_parquet(ranking_risco_path, index=False)
ranking_financeiro.to_parquet(ranking_financeiro_path, index=False)

save_report(ranking_risco.head(1000), "30_ranking_holdout_top1000_risco_perda_baseline_corrigido.csv")
save_report(ranking_financeiro.head(1000), "31_ranking_holdout_top1000_prioridade_financeira_baseline_corrigido.csv")

ranking_risco.head(20)

## 16. Classification report

In [ ]:
pred_perda_holdout_t05 = (proba_perda_holdout >= 0.5).astype(int)

print("Classification report — threshold 0.5")
print("Classe 0 = banco ganhou")
print("Classe 1 = banco perdeu")
print(classification_report(
    y_holdout,
    pred_perda_holdout_t05,
    target_names=["banco_ganhou", "banco_perdeu"]
))

print("Confusion matrix — threshold 0.5")
print("Linhas = real | Colunas = previsto")
print(confusion_matrix(y_holdout, pred_perda_holdout_t05))

## 17. Summary final

In [ ]:
summary_step_3a = pd.DataFrame([{
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "best_model_walk_forward": best_model_name,
    "best_model_mean_pr_auc_perda_wf": model_summary.loc[model_summary["model"] == best_model_name, "mean_pr_auc_perda"].iloc[0],
    "best_model_mean_roc_auc_perda_wf": model_summary.loc[model_summary["model"] == best_model_name, "mean_roc_auc_perda"].iloc[0],
    "holdout_pr_auc_perda": holdout_pr_auc_perda,
    "holdout_roc_auc_perda": holdout_roc_auc_perda,
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "qtd_features": len(feature_list),
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "ranking_risco_holdout_path": str(ranking_risco_path),
    "ranking_financeiro_holdout_path": str(ranking_financeiro_path),
}])

save_report(summary_step_3a, "32_summary_step_3a_walk_forward_baseline_corrigido.csv")
summary_step_3a.T

# O que verificar após rodar

Enviar na próxima iteração:

1. `fold_summary`
2. `model_summary`
3. `holdout_metrics`
4. `holdout_topk_perda`
5. `holdout_topk_financeiro`
6. `summary_step_3a.T`

## Interpretação correta

```text
proba_banco_perdeu = predict_proba(...)[1]
score_risco_perda = proba_banco_perdeu
score_prioridade_financeira = proba_banco_perdeu × valor_ajuizado
```